# D2C Growth & Churn Intelligence Platform — Python Analysis
RFM segmentation, cohort retention, and churn scoring, built on the cleaned `orders_clean` / `customers_clean` views exported from SQL Server.

## 1. Load Data
The exported CSVs have **no header row** — the first line is already real data (the first order/customer), not column names. `header=None, names=[...]` tells pandas not to assume row 1 is a header, and supplies the correct column names manually (known from the SQL `CREATE VIEW` statements).

In [ ]:
import pandas as pd

orders_columns = [
    "order_id", "customer_id", "is_guest_checkout", "order_date", "product_id",
    "category", "quantity", "is_bad_quantity", "unit_price", "list_price",
    "is_price_outlier", "discount_pct", "gross_revenue", "acquisition_channel"
]

customers_columns = [
    "customer_id", "is_duplicate_signup", "acquisition_channel", "city", "signup_date"
]

orders = pd.read_csv("orders_cleaned.csv", header=None, names=orders_columns)
customers = pd.read_csv("customers_cleaned.csv", header=None, names=customers_columns)

print("orders shape:", orders.shape)
print("customers shape:", customers.shape)
print(orders.head(3))

## 2. RFM — Calculate Recency, Frequency, Monetary
For every customer: **Recency** (days since last order, lower = better), **Frequency** (order count, higher = better), **Monetary** (total spend, higher = better). `reference_date` uses the latest order date in the dataset as a stand-in for "today", since this is historical, not live, data.

In [ ]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
reference_date = orders['order_date'].max()

rfm = orders[orders['is_bad_quantity'] == 0].groupby('customer_id').agg(
    recency_days=('order_date', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('gross_revenue', 'sum')
).reset_index()

print(rfm.shape)
print(rfm.head())

## 3. RFM Scoring (1–5 per metric, via quantiles)
`pd.qcut` splits customers into 5 equal-sized buckets by each metric — the best 20% get a 5, the worst 20% get a 1. Quantiles are used instead of fixed thresholds (e.g. "recency < 30 days = 5") because they adapt automatically to the actual data rather than assuming a pre-decided definition of "good". Recency scoring is reversed (`labels=[5,4,3,2,1]`) since a *lower* recency_days is better. `frequency.rank(method='first')` breaks ties (many customers share the same order count) so `qcut` always has something distinct to split on.

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['m_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5])

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

print(rfm.head(10))

## 4. Segment Customers into Business Labels
Combines R and F scores into 6 human-readable segments using standard RFM logic. M score isn't used in the rule itself — R and F together already capture the key behavioral pattern, keeping the rules simple and easy to explain.

In [ ]:
def segment_customer(row):
    r = int(row['r_score'])
    f = int(row['f_score'])

    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r <= 2 and f >= 4:
        return 'At Risk'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

print(rfm['segment'].value_counts())

**Finding:** a near-even split — 22% Champions, 21.5% Lost — meaning the customer base is almost evenly divided between the business's best customers and its most disengaged ones. A clear signal for where retention effort should focus ("At Risk" and "Needs Attention").

## 5. Visualize Segment Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

segment_order = rfm['segment'].value_counts().index

plt.figure(figsize=(8, 5))
sns.barplot(x=rfm['segment'].value_counts().values,
            y=segment_order,
            order=segment_order)
plt.title('Customer Distribution by RFM Segment')
plt.xlabel('Number of Customers')
plt.ylabel('')
plt.tight_layout()
plt.savefig('rfm_segment_distribution.png', dpi=150)
plt.show()

## 6. Cohort Retention — Prepare Data
Group customers by signup month (their "cohort"), then attach each order to its customer's cohort. `how='inner'` naturally drops guest checkout orders (no customer_id to match), since a guest can't be assigned to a cohort.

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['signup_month'] = customers['signup_date'].dt.to_period('M')

orders_with_cohort = orders.merge(
    customers[['customer_id', 'signup_month']],
    on='customer_id',
    how='inner'
)

orders_with_cohort['order_month'] = orders_with_cohort['order_date'].dt.to_period('M')

print(orders_with_cohort.shape)
print(orders_with_cohort[['customer_id', 'signup_month', 'order_month']].head())

## 7. Cohort Retention Table


The first version defined a cohort's starting size as "however many customers ordered in their signup month" (`cohort_pivot.iloc[:, 0]`). But not everyone's first order happens in their signup month — someone might sign up in December and not order until January. This undercounted the true cohort size, causing later months to show impossible retention values over 100%.

**Fix:** cohort size is redefined as the actual number of people who *signed up* that month (`customers.groupby('signup_month')`), not who happened to order immediately. Lesson: the denominator in a retention metric should represent the true starting population, not a convenient but incorrect proxy for it.

In [ ]:
orders_with_cohort['months_since_signup'] = (
    (orders_with_cohort['order_month'] - orders_with_cohort['signup_month']).apply(lambda x: x.n)
)

cohort_data = orders_with_cohort.groupby(['signup_month', 'months_since_signup'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['signup_month', 'months_since_signup', 'active_customers']

cohort_pivot = cohort_data.pivot(index='signup_month', columns='months_since_signup', values='active_customers')

# correct cohort size: actual signups that month, not "who ordered in month 0"
actual_cohort_sizes = customers.groupby('signup_month')['customer_id'].nunique()

retention_pct = cohort_pivot.divide(actual_cohort_sizes, axis=0) * 100

print(retention_pct.round(1).iloc[:8, :7])

**Finding:** month-0 retention averages ~35–40%, decaying to single digits by month 5–6 — a believable, typical D2C retention decay curve.

## 8. Save RFM and Cohort Outputs

In [ ]:
retention_pct.to_csv('cohort_retention.csv')
rfm.to_csv('rfm_output.csv', index=False)

print("Both files saved.")
print("rfm columns:", list(rfm.columns))

## 9. Churn Scoring
Using the 104.9-day average reorder gap found in SQL, a churn threshold of **210 days** (roughly 2x the average) flags any customer whose most recent order is older than that.

In [ ]:
churn_threshold = 210  # roughly 2x the average 104.9-day reorder gap (from SQL analysis)

rfm['is_churned'] = rfm['recency_days'] > churn_threshold

churn_summary = rfm.groupby('segment')['is_churned'].agg(['sum', 'count'])
churn_summary['churn_rate_pct'] = (churn_summary['sum'] / churn_summary['count'] * 100).round(1)

print(churn_summary)

**Finding:** Champions show ~4% churn, while Lost/At Risk/Needs Attention all show ~100% — confirming the segmentation and churn logic agree with each other, a good internal consistency check.

## 10. Final Save (with churn flag included)

In [ ]:
rfm.to_csv('rfm_output.csv', index=False)
print("Final RFM saved:", rfm.shape)